# Model B - Lightweight CNN (EuroSAT)

EN3150 Assignment 03. Model B will use depthwise-separable convolutions with at most **100,000 trainable parameters**.

This notebook establishes the shared data pipeline and untrained lightweight architecture. Parameter and MAC analysis is included below; optimizer experiments, training and evaluation will be added in subsequent commits.

## Data contract

Run `Dataset.ipynb` once before this notebook, from the repository root. We reuse `data/eurosat_split.csv` and `data/eurosat/train|val|test/<class>/` without downloading, modifying or re-splitting images.

- 27,000 native 64x64 RGB images; 10 alphabetically ordered classes.
- Stratified 70/15/15 split: 18,900 training, 4,050 validation, 4,050 test images.
- Pixels scaled from 0-255 to float32 in 0-1; no augmentation.
- Training order changes each epoch. Validation and test order remain fixed.
- Test files are checked for structural consistency here; no test predictions or model selection occur.


In [9]:
from pathlib import Path
import hashlib
import platform

import numpy as np
import pandas as pd
import tensorflow as tf

DATA_DIR = Path("data/eurosat")
SPLIT_CSV = Path("data/eurosat_split.csv")
IMG_SIZE = (64, 64)
SEED = 42
BATCH_SIZE = 128  # Match Model A's final training batch size.
AUTOTUNE = tf.data.AUTOTUNE
CLASS_NAMES = [
    "AnnualCrop", "Forest", "HerbaceousVegetation", "Highway", "Industrial",
    "Pasture", "PermanentCrop", "Residential", "River", "SeaLake",
]
NUM_CLASSES = len(CLASS_NAMES)
EXPECTED_COUNTS = {"train": 18_900, "val": 4_050, "test": 4_050}

tf.keras.utils.set_random_seed(SEED)
print("Python:", platform.python_version(), "| TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "none")


Python: 3.12.2 | TensorFlow: 2.21.0
GPU: none


## 1. Verify the committed split and local files

Fail early if local folders differ from the shared manifest. Image identities are `(label, filename)` pairs. Checks cover duplicate identities, allowed labels/splits, stratification, and exact file membership. The manifest hash identifies the split used by future experiments.


In [10]:
assert SPLIT_CSV.is_file() and DATA_DIR.is_dir(), (
    "Shared data is missing. Run Dataset.ipynb from the repository root first."
)
manifest = pd.read_csv(SPLIT_CSV, dtype=str)
assert list(manifest.columns) == ["file", "label", "split"], "Unexpected manifest columns"
assert not manifest.isna().any().any(), "Manifest contains missing values"
assert not manifest.duplicated(["label", "file"]).any(), "An image appears more than once in the manifest"
assert set(manifest["label"]) == set(CLASS_NAMES), "Unexpected classes"
assert set(manifest["split"]) == set(EXPECTED_COUNTS), "Unexpected split names"
assert manifest["file"].map(lambda name: Path(name).name == name and name.lower().endswith(".jpg")).all(), "Invalid image filenames"
assert len(manifest) == 27_000, "Expected 27,000 images"

counts = pd.crosstab(manifest["label"], manifest["split"]).reindex(
    index=CLASS_NAMES, columns=list(EXPECTED_COUNTS)
)
assert not counts.isna().any().any(), "A class is missing from a split"
np.testing.assert_allclose(
    counts.to_numpy() / counts.sum(axis=1).to_numpy()[:, None],
    np.tile([0.70, 0.15, 0.15], (NUM_CLASSES, 1)), rtol=0, atol=1e-12,
)

expected_paths = {
    f"{row.split}/{row.label}/{row.file}" for row in manifest.itertuples(index=False)
}
actual_paths = {p.relative_to(DATA_DIR).as_posix() for p in DATA_DIR.rglob("*") if p.is_file()}
missing = sorted(expected_paths - actual_paths)
extra = sorted(actual_paths - expected_paths)
assert not missing and not extra, f"Dataset differs from manifest: missing={missing[:5]}, extra={extra[:5]}"
for split, expected in EXPECTED_COUNTS.items():
    assert int(counts[split].sum()) == expected, f"Wrong count for {split}"

SPLIT_SHA256 = hashlib.sha256(SPLIT_CSV.read_bytes()).hexdigest()
print(counts.to_string())
print("Totals:", counts.sum().to_dict())
print("Split SHA-256:", SPLIT_SHA256)
print("Manifest and local file membership agree.")


split                 train  val  test
label                                 
AnnualCrop             2100  450   450
Forest                 2100  450   450
HerbaceousVegetation   2100  450   450
Highway                1750  375   375
Industrial             1750  375   375
Pasture                1400  300   300
PermanentCrop          1750  375   375
Residential            2100  450   450
River                  1750  375   375
SeaLake                2100  450   450
Totals: {'train': 18900, 'val': 4050, 'test': 4050}
Split SHA-256: 0cf4cc1ca5601a80a4a9c7ce47fb3435fcf95a36eba01802e0d08baf719f0c3c
Manifest and local file membership agree.


## 2. Load and normalise images

Read JPEGs directly instead of resizing them: an unexpected size or channel count must fail rather than be silently corrected. Validate decoded files in all splits once, without caching their pixels in RAM. Each float32 image occupies about 48 KiB, so caching all 27,000 images would require about 1.24 GiB before pipeline overhead.

Shuffle file paths before decoding, then batch and prefetch. Recreating the training dataset with the same seed gives each future optimizer experiment the same initial shuffle state. Validation and test paths are sorted and never shuffled.


In [11]:
def decode_image(path, label):
    image = tf.io.decode_jpeg(tf.io.read_file(path), channels=0)
    image = tf.ensure_shape(image, IMG_SIZE + (3,))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label


split_paths = {}
split_labels = {}
for split in EXPECTED_COUNTS:
    rows = manifest.loc[manifest["split"] == split].sort_values(["label", "file"])
    split_paths[split] = [str(DATA_DIR / row.split / row.label / row.file) for row in rows.itertuples(index=False)]
    split_labels[split] = np.array([CLASS_NAMES.index(label) for label in rows["label"]], dtype=np.int32)


def make_dataset(split, batch_size=BATCH_SIZE):
    """Create a fresh deterministic pipeline; only training is shuffled each epoch."""
    if split not in EXPECTED_COUNTS:
        raise ValueError(f"Unknown split: {split}")
    ds = tf.data.Dataset.from_tensor_slices((split_paths[split], split_labels[split]))
    if split == "train":
        ds = ds.shuffle(EXPECTED_COUNTS[split], seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(decode_image, num_parallel_calls=AUTOTUNE, deterministic=True)
    return ds.batch(batch_size).prefetch(AUTOTUNE)


print("Class mapping:", dict(enumerate(CLASS_NAMES)))
# Consume every image once to surface malformed JPEGs and wrong shapes now.
# Validation is structural only; no model or test-performance metric is involved.
for split in EXPECTED_COUNTS:
    seen = 0
    for images, labels in make_dataset(split):
        tf.debugging.assert_type(images, tf.float32)
        tf.debugging.assert_all_finite(images, "Non-finite pixels")
        tf.debugging.assert_greater_equal(images, 0.0)
        tf.debugging.assert_less_equal(images, 1.0)
        tf.debugging.assert_greater_equal(labels, 0)
        tf.debugging.assert_less(labels, NUM_CLASSES)
        seen += int(tf.shape(images)[0])
    assert seen == EXPECTED_COUNTS[split], f"Wrong decoded count for {split}"
    print(f"{split}: {seen:,} decoded 64x64 RGB images validated")

# Fresh pipelines after validation; the training shuffle has not been consumed.
train_ds = make_dataset("train")
val_ds = make_dataset("val")
test_ds = make_dataset("test")


Class mapping: {0: 'AnnualCrop', 1: 'Forest', 2: 'HerbaceousVegetation', 3: 'Highway', 4: 'Industrial', 5: 'Pasture', 6: 'PermanentCrop', 7: 'Residential', 8: 'River', 9: 'SeaLake'}
train: 18,900 decoded 64x64 RGB images validated
val: 4,050 decoded 64x64 RGB images validated
test: 4,050 decoded 64x64 RGB images validated


## 3. Lightweight architecture

Use the same four-block structure as Model A, replacing each standard convolution with a bias-free `SeparableConv2D`: a 3x3 depthwise filter per input channel followed by a 1x1 pointwise convolution. Depth multiplier is 1. BatchNorm, ReLU and 2x2 max pooling follow each separable convolution.

Filters are 32, 64, 128 and 256. Pooling reduces spatial dimensions from 64 to 4. Global average pooling produces 256 features, followed by Dense(128, ReLU), Dropout(0.5) and Dense(10, softmax). This retains Model A's head and regularisation for an interpretable architectural comparison. All weights start from scratch.

The assignment limit applies to **trainable parameters**. Report total parameters separately because BatchNorm also stores non-trainable moving statistics. Layer-by-layer formulas and MAC analysis follow in section 4.


In [12]:
from tensorflow.keras import layers, models


def build_model_b(input_shape=IMG_SIZE + (3,), num_classes=NUM_CLASSES):
    """Build a fresh depthwise-separable CNN without compiling or training it."""
    inputs = layers.Input(shape=input_shape, name="image")
    x = inputs
    for block, filters in enumerate([32, 64, 128, 256], start=1):
        x = layers.SeparableConv2D(
            filters, kernel_size=3, padding="same", depth_multiplier=1,
            use_bias=False, name=f"block_{block}_separable_conv",
        )(x)
        x = layers.BatchNormalization(name=f"block_{block}_bn")(x)
        x = layers.ReLU(name=f"block_{block}_relu")(x)
        x = layers.MaxPooling2D(pool_size=2, name=f"block_{block}_pool")(x)
    x = layers.GlobalAveragePooling2D(name="global_average_pool")(x)
    x = layers.Dense(128, activation="relu", name="classifier_hidden")(x)
    x = layers.Dropout(0.5, name="classifier_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)
    return models.Model(inputs, outputs, name="Model_B_LightweightCNN")


tf.keras.utils.set_random_seed(SEED)
model_b = build_model_b()
model_b.summary()


Model: "Model_B_LightweightCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ image (InputLayer)              │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_1_separable_conv          │ (None, 64, 64, 32)     │           123 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_1_bn (BatchNormalization) │ (None, 64, 64, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_1_relu (ReLU)             │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_1_pool (MaxPooling2D)     │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_2_separable_conv          │ (None, 32, 32, 64)     │         2,336 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_2_bn (BatchNormalization) │ (None, 32, 32, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_2_relu (ReLU)             │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_2_pool (MaxPooling2D)     │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_3_separable_conv          │ (None, 16, 16, 128)    │         8,768 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_3_bn (BatchNormalization) │ (None, 16, 16, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_3_relu (ReLU)             │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_3_pool (MaxPooling2D)     │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_4_separable_conv          │ (None, 8, 8, 256)      │        33,920 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_4_bn (BatchNormalization) │ (None, 8, 8, 256)      │         1,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_4_relu (ReLU)             │ (None, 8, 8, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block_4_pool (MaxPooling2D)     │ (None, 4, 4, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pool             │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier_hidden (Dense)       │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier_dropout (Dropout)    │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_probabilities (Dense)     │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 81,253 (317.39 KB)

 Trainable params: 80,293 (313.64 KB)

 Non-trainable params: 960 (3.75 KB)

In [13]:
n_trainable = int(sum(np.prod(weight.shape) for weight in model_b.trainable_weights))
n_non_trainable = int(sum(np.prod(weight.shape) for weight in model_b.non_trainable_weights))
n_total = int(model_b.count_params())
assert n_trainable <= 100_000, f"Trainable parameter budget exceeded: {n_trainable:,}"
assert n_total == n_trainable + n_non_trainable
assert model_b.input_shape == (None, 64, 64, 3)
assert model_b.output_shape == (None, NUM_CLASSES)

# A validation batch checks the interface; no test predictions or performance metrics.
x_batch, y_batch = next(iter(make_dataset("val", batch_size=4)))
probabilities = model_b(x_batch, training=False).numpy()
assert probabilities.shape == (len(x_batch), NUM_CLASSES)
assert np.isfinite(probabilities).all(), "Non-finite output probabilities"
assert ((probabilities >= 0) & (probabilities <= 1)).all()
np.testing.assert_allclose(probabilities.sum(axis=1), 1.0, rtol=0, atol=1e-5)

# The factory must return independent layers and weights for future optimizer runs.
fresh_model_b = build_model_b()
assert fresh_model_b is not model_b
assert fresh_model_b.count_params() == n_total
assert not ({id(w) for w in model_b.weights} & {id(w) for w in fresh_model_b.weights})
del fresh_model_b

print(f"Trainable parameters     : {n_trainable:,} / 100,000")
print(f"Non-trainable parameters : {n_non_trainable:,}")
print(f"Total parameters         : {n_total:,}")
print(f"Output shape            : {probabilities.shape}")
print("Architecture checks passed. This model is untrained.")


Trainable parameters     : 80,293 / 100,000
Non-trainable parameters : 960
Total parameters         : 81,253
Output shape            : (4, 10)
Architecture checks passed. This model is untrained.


## 4. Parameters and multiply-accumulate operations (MACs)

For a bias-free separable convolution with depth multiplier 1:

| Operation | Parameters | MACs per image |
|---|---|---|
| Depthwise k x k | k*k*Cin | Hout*Wout*k*k*Cin |
| Pointwise 1 x 1 | Cin*Cout | Hout*Wout*Cin*Cout |
| BatchNorm | 4*Cout total; 2*Cout trainable | Excluded from this MAC convention |
| Dense | Nin*Nout + Nout biases | Nin*Nout |
| ReLU, max pooling, global average pooling, dropout | 0 | Excluded |

BatchNorm stores trainable scale/shift and non-trainable moving mean/variance. The counts below include all weights but count MACs only for convolution and dense operations, matching Model A's convention. They exclude bias additions, normalisation, pooling, activation and softmax operations, so are not a complete instruction count or a latency measurement. Deployment tooling may fold BatchNorm into convolutions; this notebook does not perform that transformation.

The table includes every layer and separates depthwise and pointwise costs. Its formula totals must match Keras, including non-trainable statistics.


In [14]:
def model_b_layer_table(model):
    rows = []
    for layer in model.layers:
        depthwise_params = pointwise_params = 0
        depthwise_macs = pointwise_macs = 0
        kernel = ""
        filters_units = ""
        params = trainable = macs = 0
        if isinstance(layer, layers.SeparableConv2D):
            kh, kw = layer.kernel_size
            h, w, cout = map(int, layer.output.shape[1:])
            cin = int(layer.input.shape[-1])
            multiplier = int(layer.depth_multiplier)
            depthwise_params = kh * kw * cin * multiplier
            pointwise_params = cin * multiplier * cout
            params = trainable = depthwise_params + pointwise_params + (cout if layer.use_bias else 0)
            depthwise_macs = h * w * depthwise_params
            pointwise_macs = h * w * pointwise_params
            macs = depthwise_macs + pointwise_macs
            kernel, filters_units = f"{kh}x{kw}", cout
        elif isinstance(layer, layers.BatchNormalization):
            channels = int(layer.input.shape[-1])
            assert layer.scale and layer.center, "Formula assumes default BatchNorm scale and center"
            params, trainable = 4 * channels, 2 * channels
        elif isinstance(layer, layers.Dense):
            nin, nout = int(layer.input.shape[-1]), int(layer.units)
            params = trainable = nin * nout + (nout if layer.use_bias else 0)
            macs, filters_units = nin * nout, nout
        elif not isinstance(layer, (layers.InputLayer, layers.ReLU, layers.MaxPooling2D,
                                    layers.GlobalAveragePooling2D, layers.Dropout)):
            raise TypeError(f"Missing cost formula for {type(layer).__name__}")
        keras_trainable = int(sum(np.prod(w.shape) for w in layer.trainable_weights))
        assert params == layer.count_params(), f"Parameter mismatch: {layer.name}"
        assert trainable == keras_trainable, f"Trainable mismatch: {layer.name}"
        rows.append(dict(
            layer=layer.name, type=type(layer).__name__, kernel=kernel, filters_units=filters_units,
            output="x".join(str(d) for d in layer.output.shape[1:]),
            params=params, trainable=trainable, non_trainable=params - trainable,
            depthwise_params=depthwise_params, pointwise_params=pointwise_params,
            depthwise_macs=depthwise_macs, pointwise_macs=pointwise_macs, macs=macs,
        ))
    return pd.DataFrame(rows)


cost_table = model_b_layer_table(model_b)
cost_totals = cost_table[["params", "trainable", "non_trainable", "macs"]].sum()
assert int(cost_totals["params"]) == model_b.count_params()
assert int(cost_totals["trainable"]) == n_trainable
assert int(cost_totals["non_trainable"]) == n_non_trainable
assert int(cost_totals["trainable"]) <= 100_000
MODEL_B_MACS = int(cost_totals["macs"])
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
cost_table.to_csv(RESULTS_DIR / "model_b_layers.csv", index=False)
print(cost_table.to_string(index=False))
print("\nVerified totals:", {key: int(value) for key, value in cost_totals.items()})
print(f"Model B MACs per 64x64 image: {MODEL_B_MACS:,}")


                 layer                   type kernel filters_units    output  params  trainable  non_trainable  depthwise_params  pointwise_params  depthwise_macs  pointwise_macs    macs
                 image             InputLayer                        64x64x3       0          0              0                 0                 0               0               0       0
block_1_separable_conv        SeparableConv2D    3x3            32  64x64x32     123        123              0                27                96          110592          393216  503808
            block_1_bn     BatchNormalization                       64x64x32     128         64             64                 0                 0               0               0       0
          block_1_relu                   ReLU                       64x64x32       0          0              0                 0                 0               0               0       0
          block_1_pool           MaxPooling2D                    

## 5. Design and hardware-aware justification

**Separable convolutions.** Standard 3x3 convolution uses 9*Cin*Cout weights; our separable version uses 9*Cin + Cin*Cout. The same ratio applies to convolution MACs at a fixed output size. For block 4 (128 to 256 channels), this reduces convolution weights from 294,912 to 33,920, about 8.69 times fewer. The reduction depends on channel counts; depthwise and pointwise stages restrict how spatial filtering and channel mixing are represented, which may affect accuracy. Only training and evaluation can establish that trade-off.

**Four blocks, 3x3 kernels and 32/64/128/256 channels.** Retain Model A's feature-map dimensions and filter counts so the main architectural change is separable filtering. Small kernels capture local texture; successive blocks combine wider context while pooling reduces spatial work. Doubling channels compensates partly for reduced spatial resolution. A 5x5 depthwise kernel would increase the spatial-filter term from 9*Cin to 25*Cin, while leaving the pointwise term unchanged.

**Global average pooling and classifier.** Averaging the 4x4x256 map gives 256 features. Dense(128) therefore needs 32,896 parameters, instead of 524,416 after flattening. Keep Model A's Dense(128), dropout 0.5 and ten-class softmax to control the comparison. The dense head becomes a significant part of this smaller model; a smaller head is a possible validation-driven experiment, not part of this starting architecture.

**BatchNorm and omitted convolution biases.** Normalisation adds learned scale and shift plus moving statistics; an extra preceding bias is unnecessary. Inference-time folding is possible with suitable deployment tooling, but the saved Keras architecture retains BatchNorm unless explicitly converted.

**ReLU.** `max(0, x)` needs a comparison and avoids exponentials in hidden layers. It is compatible with quantised inference and can be fused by supporting inference engines. Its derivative on positive inputs is 1, although inactive units can still have zero gradients. Actual speed depends on the hardware and runtime; sparsity alone does not guarantee faster execution.

**Softmax.** Apply it only to the ten output scores for multiclass cross-entropy and probabilities. If deployment needs only the winning class, argmax of logits gives the same class, but removing softmax would require an explicit deployment change.

**Scope of these estimates.** Fewer weights reduce raw weight storage; fewer MACs indicate less arithmetic. Intermediate activations, memory traffic and runtime support also affect peak memory and latency. No device-speed, quantisation or accuracy claim follows from these counts alone.


## 6. Reproducible optimizer experiment workflow

Each run resets the seed, builds fresh weights and creates a fresh training pipeline. Compare optimizers with the same batch size, epoch budget and constant learning rate; no early stopping or learning-rate schedules during this comparison. Test data is never used.

Save each completed run separately with architecture, split-manifest and runtime metadata. A matching cache can be reused; changed metadata requires explicit replacement. Files are written atomically so an interrupted write does not replace a completed run. Timing includes training and validation, and the reported mean excludes epoch 1 when possible.

The smoke run below is opt-in and uses only eight training and eight validation images for one epoch, batch size 4. It checks execution and cache behavior; it provides no evidence for optimizer selection or final accuracy. Full experiments are a later step.


In [15]:
import json
import os
import time

EXPERIMENT_SCHEMA = 1
PIPELINE_VERSION = "eurosat-rgb-unit-scale-shuffle-paths-v1"
EXPERIMENT_DIR = Path("results/model_b_experiments")


class EpochTimer(tf.keras.callbacks.Callback):
    def on_train_begin(self, logs=None):
        self.times = []

    def on_epoch_begin(self, epoch, logs=None):
        self.started = time.perf_counter()

    def on_epoch_end(self, epoch, logs=None):
        self.times.append(float(time.perf_counter() - self.started))


def make_optimizer(name, learning_rate, momentum=0.0):
    if learning_rate <= 0 or not 0 <= momentum < 1:
        raise ValueError("Require positive learning rate and momentum in [0, 1)")
    if name == "adam" and momentum == 0:
        return tf.keras.optimizers.Adam(learning_rate=learning_rate)
    if name == "sgd":
        return tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum)
    raise ValueError("Use 'adam' with momentum=0, or 'sgd'")


def architecture_signature(model):
    # Explicit layer names and a fresh build make this independent of generated names.
    config = json.loads(model.to_json())
    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()


def save_experiment(path, result):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(result, indent=2, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def run_optimizer_experiment(run_id, optimizer, learning_rate, momentum=0.0,
                             epochs=20, batch_size=BATCH_SIZE, smoke=False,
                             rerun=False, output_dir=EXPERIMENT_DIR, verbose=2):
    """Run or reuse one complete validation-only experiment; never access test_ds."""
    if not run_id or any(c not in "abcdefghijklmnopqrstuvwxyz0123456789_-" for c in run_id):
        raise ValueError("run_id must use lowercase letters, digits, underscores or hyphens")
    if epochs < 1 or batch_size < 1:
        raise ValueError("epochs and batch_size must be positive")
    if smoke and (epochs != 1 or batch_size > 4):
        raise ValueError("Smoke runs require one epoch and batch size at most 4")
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    candidate = build_model_b()
    opt = make_optimizer(optimizer, learning_rate, momentum)
    trainable = int(sum(np.prod(w.shape) for w in candidate.trainable_weights))
    assert trainable <= 100_000
    metadata = {
        "schema": EXPERIMENT_SCHEMA, "pipeline": PIPELINE_VERSION,
        "architecture_sha256": architecture_signature(candidate),
        "split_sha256": SPLIT_SHA256, "class_names": CLASS_NAMES,
        "seed": SEED, "batch_size": int(batch_size), "epochs": int(epochs),
        "optimizer": optimizer, "learning_rate": float(learning_rate),
        "momentum": float(momentum), "smoke": bool(smoke),
        "sample_counts": {"train": 8, "val": 8} if smoke else {
            key: EXPECTED_COUNTS[key] for key in ["train", "val"]},
        "python": platform.python_version(), "tensorflow": tf.__version__,
        "keras": tf.keras.__version__, "numpy": np.__version__,
        "platform": platform.platform(), "processor": platform.processor(),
        "devices": [device.name for device in tf.config.list_physical_devices()],
    }
    path = Path(output_dir) / f"{run_id}.json"
    if path.exists() and not rerun:
        cached = json.loads(path.read_text(encoding="utf-8"))
        if cached.get("metadata") != metadata:
            raise ValueError(f"Cache metadata differs for {path}; choose a new run ID or set rerun=True")
        history = cached.get("history", {})
        if any(len(history.get(key, [])) != epochs for key in
               ["loss", "accuracy", "val_loss", "val_accuracy"]) or len(cached.get("epoch_times_s", [])) != epochs:
            raise ValueError(f"Incomplete cache: {path}")
        print("Reusing completed experiment:", path)
        return cached

    def experiment_data(split):
        if not smoke:
            return make_dataset(split, batch_size=batch_size)
        # Limit paths before decoding or prefetching, keeping the smoke run small.
        ds = tf.data.Dataset.from_tensor_slices((split_paths[split][:8], split_labels[split][:8]))
        if split == "train":
            ds = ds.shuffle(8, seed=SEED, reshuffle_each_iteration=True)
        return ds.map(decode_image, num_parallel_calls=1).batch(batch_size).prefetch(1)

    candidate.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    timer = EpochTimer()
    history = candidate.fit(experiment_data("train"), validation_data=experiment_data("val"),
                            epochs=epochs, callbacks=[timer], verbose=verbose).history
    history = {key: [float(value) for value in history[key]] for key in
               ["loss", "accuracy", "val_loss", "val_accuracy"]}
    if any(len(values) != epochs or not np.isfinite(values).all() for values in history.values()):
        raise ValueError("Experiment did not complete with finite metrics; nothing saved")
    best = int(np.argmax(history["val_accuracy"]))
    result = {
        "metadata": metadata, "history": history, "epoch_times_s": timer.times,
        "summary": {"best_val_accuracy": history["val_accuracy"][best], "best_epoch": best + 1,
                    "final_val_accuracy": history["val_accuracy"][-1],
                    "final_val_loss": history["val_loss"][-1],
                    "time_per_epoch_s": float(np.mean(timer.times[1:] or timer.times)),
                    "timing_excludes_first_epoch": len(timer.times) > 1},
    }
    save_experiment(path, result)
    print("Saved completed experiment:", path)
    return result


In [16]:
RUN_SMOKE = False  # Set True to verify training and cache handling locally.
if RUN_SMOKE:
    import tempfile

    with tempfile.TemporaryDirectory(prefix="model_b_smoke_") as temporary_dir:
        settings = dict(run_id="smoke", optimizer="adam", learning_rate=0.001,
                        epochs=1, batch_size=4, smoke=True, output_dir=Path(temporary_dir), verbose=0)
        smoke_result = run_optimizer_experiment(**settings)
        cached_result = run_optimizer_experiment(**settings)
        assert smoke_result == cached_result
        try:
            run_optimizer_experiment(**{**settings, "learning_rate": 0.003})
        except ValueError as error:
            assert "Cache metadata differs" in str(error)
        else:
            raise AssertionError("Changed experiment settings must reject the cache")
        assert not list(Path(temporary_dir).glob("*.tmp"))
    print("Smoke run, cache reuse and cache mismatch checks passed. No experiment artifacts retained.")


## Next stage

Define and run the full optimizer comparison, save its histories and plots, and select a configuration using validation results. The workflow above does not launch full experiments automatically.
